# 文献调查研究论文摘要

进行文献调查意味着浏览数十篇研究论文，找出哪些论文真正与您的主题相关——这可能需要数小时甚至数天的阅读时间。该工具通过分析每个 PDF、提取关键信息并生成带有相关性分数和颜色编码的结构化 Excel 表格来自动化该过程，以便您可以立即查看哪些论文值得深入阅读。

## 这个工具的作用
- 读取文件夹中的所有 PDF 研究论文
- 提取关键字段：目标、方法、验证方法、标准、限制等
- 根据与您的研究主题的相关性对每篇论文进行评分 (1-10)
- 生成颜色编码的 Excel 表格 — 绿色（高度相关）、黄色（中等）、红色（低相关）
- 自动将论文从最相关到最不相关进行排序

## 如何使用
1. 将 PDF 论文添加到与此笔记本同一目录中名为“papers/”的文件夹中
2. 用您的研究领域更新下面的“research_topic”变量
3. 运行所有单元格
4.查看同目录下生成的输出Excel文件

## 要求
- `.env` 文件中的 OpenAI API 密钥
- 安装依赖项：`uv pip install pdfplumber pandas openpyxl openai python-dotenv`

---
*作为 Udemy 课程的第一天扩展项目构建：AI 工程师核心课程 — LLM 工程、RAG、QLoRA、Agents，作者：Ed Donner。*

### 安装所需的库
在运行笔记本的其余部分之前运行下面的单元格：
- `pdfplumber` — 从 PDF 文件中提取文本
- `pandas` — 创建并操作数据表
- `openpyxl` — 创建并格式化 Excel 输出文件
- `openai` — 连接到 OpenAI API 来分析论文
- `python-dotenv` — 从 .env 文件安全加载您的 OpenAI API 密钥

In [ ]:
!uv pip install pdfplumber pandas openpyxl openai python-dotenv

In [ ]:
# Standard library 导入
import os
import json

# Third party 导入
import pdfplumber          # extracts text from PDF files
import pandas as pd        # creates and manipulates the data table
from openai import OpenAI  # connects to the OpenAI API
from openpyxl import load_workbook                    # opens and edits Excel files
from openpyxl.styles import PatternFill, Alignment    # formats Excel cells
from dotenv import load_dotenv                        # loads API key from .env file


In [ ]:
# Load the OpenAI API 密钥 securely from the .env file
load_dotenv(override=True)

In [ ]:
# 初始化 the OpenAI client — this is what we use to make API calls throughout the notebook
openai = OpenAI()

In [ ]:
# 将其更改为您的研究主题

research_topic = """
UAV intelligence quality assurance, standards, and validation methods. 
Topics of interest include: UAV system reliability, fault detection, 
testing frameworks, quality standards, validation methodologies, 
and intelligent UAV systems.
"""

# 系统提示指示 LLM 如何行事以及提取什么
# 我们传递研究主题，以便它能够准确地判断相关性
# LLM 被要求以 JSON 形式响应，以便我们可以将其解析为结构化表

system_prompt = f"""
You are a research assistant helping with a literature survey on this topic: {research_topic}

Extract the following fields from the research paper and respond ONLY in valid JSON format with these exact keys:
{{
    "Year": "Look carefully for the publication year in the copyright notice, journal header, submission date, or first page. Return only the 4-digit year. If truly not found, write Unknown",
    "Paper Title": "",
    "Authors": "",
    "Application Domain": "",
    "AI / Intelligence Component": "",
    "Objective": "",
    "Validation Method": "",
    "Test Environment": "",
    "Evaluation Metrics": "",
    "Robustness / Safety Testing": "",
    "Standards Mentioned": "",
    "Standards Body Referenced": "",
    "Limitations": "",
    "Is Relevant": "Yes or No only",
    "Relevance Score": "Rate strictly from 1 to 10 based on how directly the paper addresses UAV intelligence quality assurance, validation standards, or certification methods. 9-10: paper directly addresses UAV QA frameworks, validation standards, or certification as its PRIMARY contribution. 7-8: paper covers UAV fault detection, reliability, or safety testing but QA/standards is not the main focus. 5-6: paper uses UAVs as a tool for another application like inspection, agriculture, or mapping with minimal QA focus. 1-4: paper has little or no connection to UAV QA or validation standards."
}}
"""

# 从 PDF 的每一页中提取所有文本
# 如果您的模型具有较小的上下文窗口，请通过添加以下内容来限制文本： return text[:15000]
def extract_text_from_pdf(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        text = "\n".join(page.extract_text() for page in pdf.pages if page.extract_text())
    return text  

# 将提取的 PDF 文本发送到 OpenAI 并获取结构化 JSON
# 然后 JSON 被解析为 Python 字典，以便于表转换
def analyze_paper(pdf_path):
    text = extract_text_from_pdf(pdf_path)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Here is the research paper:\n\n{text}"}
    ]
    response = openai.chat.completions.create(
        model="gpt-5-nano",   # fastest and cheapest GPT-5 model, great for summarization
        messages=messages
    )
    raw = response.choices[0].message.content
    # 去除模型可能在 JSON 周围添加的任何 Markdown 格式
    raw = raw.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(raw)


# 将其更改为您的论文文件夹路径
# 将所有 PDF 研究论文放入此文件夹中

papers_folder = "papers/"
results = []

# 循环浏览文件夹中的所有 PDF 并分析每一个
for filename in sorted(os.listdir(papers_folder)):
    if filename.endswith(".pdf"):
        print(f"Processing: {filename}")
        pdf_path = os.path.join(papers_folder, filename)
        try:
            data = analyze_paper(pdf_path)
            data["Filename"] = filename  # track which PDF each row came from
            results.append(data)
        except Exception as e:
            print(f"Error with {filename}: {e}")


# 将结果转换为 pandas DataFrame
df = pd.DataFrame(results)

# 将 is_relevant 标准化为始终为 Yes 或 No
def standardize_relevant(val):
    if str(val).lower() in ["true", "yes", "1"]:
        return "Yes"
    elif str(val).lower() in ["false", "no", "0"]:
        return "No"
    return "Yes"

df["Is Relevant"] = df["Is Relevant"].apply(standardize_relevant)

# 将相关性得分标准化为 10 分
# LLM 有时会返回 0-1 范围而不是 1-10，因此我们对其进行标准化

def standardize_score(val):
    try:
        score = float(val)
        if score <= 1.0:  # convert 0-1 scale to 0-10
            return round(score * 10, 1)
        return round(score, 1)
    except:
        return None

df["Relevance Score"] = df["Relevance Score"].apply(standardize_score)

# 从最相关到​​最不相关对论文进行排序
df = df.sort_values("Relevance Score", ascending=False).reset_index(drop=True)

# 保存到 Excel
output_file = "literature_survey_output.xlsx"
df.to_excel(output_file, index=False)

# 对 Excel 文件应用颜色编码和格式设置
wb = load_workbook(output_file)
ws = wb.active

# 颜色定义：绿色=高度相关，黄色=中等，红色=低相关
green  = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
yellow = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")
red    = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")

headers = [cell.value for cell in ws[1]]
score_col = headers.index("Relevance Score") + 1

# 保持标题行白色且无色
for cell in ws[1]:
    cell.fill = PatternFill(fill_type=None)

# 根据每个数据行的相关性得分对每个数据行进行颜色编码
for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    score_cell = row[score_col - 1]
    try:
        score = float(score_cell.value)
        if score >= 8:
            fill = green
        elif score >= 6:
            fill = yellow
        else:
            fill = red
    except:
        fill = yellow
    for cell in row:
        cell.fill = fill

# 文本换行并将内容对齐到顶部，以提高 Excel 中的可读性
for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    for cell in row:
        cell.alignment = Alignment(wrap_text=True, vertical="top")

# 根据每列中最长的内容自动调整列宽
# 短列保持紧凑，长列的宽度上限为 60
for col in ws.columns:
    max_length = 0
    col_letter = col[0].column_letter
    for cell in col:
        try:
            if cell.value:
                cell_length = len(str(cell.value))
                if cell_length > max_length:
                    max_length = cell_length
        except:
            pass
    if max_length < 15:
        adjusted_width = max_length + 4
    elif max_length < 50:
        adjusted_width = max_length + 2
    else:
        adjusted_width = 60
    ws.column_dimensions[col_letter].width = adjusted_width

# 保存最终格式化的 Excel 文件
wb.save(output_file)
print(f"Done! Check {output_file}")